# Задача 1. EDA продаж «Поступашки»

Разбор `data/base.xlsx`: что считать покупкой, заказом и уникальным
покупателем, как обрабатывать пакеты, какие есть временные закономерности
и что из этих данных принципиально нельзя узнать.

Полный текстовый разбор со всеми деталями — в
[`reports/EDA_report.md`](../reports/EDA_report.md).

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Загрузка и очистка данных

In [2]:
# Основной путь - файл в data/base.xlsx
df = pd.read_excel("../data/base.xlsx")
df.columns = ["student_id", "amount", "course", "ts"]

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["student_id"] = pd.to_numeric(df["student_id"], errors="coerce").astype("Int64")
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["date"] = df["ts"].dt.normalize()

df.head()

,student_id,amount,course,ts,date
0,437,8950.0,ML про,2026-08-04 09:08:31,2026-08-04
1,437,7890.0,Аналитика старт,2026-08-04 09:35:00,2026-08-04
2,443,8475.0,Линейная алгебра,2026-08-04 10:14:38,2026-08-04
3,443,8475.0,Мат анализ,2026-08-04 10:14:38,2026-08-04
4,430,8950.0,К ВУЗу,2026-08-04 13:38:06,2026-08-04


## 2. Тесты качества данных

In [9]:
def run_data_tests(df: pd.DataFrame):

    null_counts = df[["student_id", "amount", "course", "ts"]].isnull().sum()
    assert null_counts.sum() == 0, f"Обнаружены пропуски:\n{null_counts}"
    print("[PASS] Пропусков нет.")

    assert (df["amount"] > 0).all(), "Обнаружены суммы <= 0!"
    print("[PASS] Все суммы положительные.")

    assert df["ts"].notnull().all(), "Не удалось распарсить часть дат!"
    print(f"[PASS] Даты валидны ({df['ts'].min()} — {df['ts'].max()}).")

    exact_dupes = df.duplicated(subset=["student_id", "course", "ts"]).sum()
    assert exact_dupes == 0, f"Найдено {exact_dupes} полных дублей строк!"
    print("[PASS] Полных дублей строк нет.")

run_data_tests(df)

[PASS] Пропусков нет.
[PASS] Все суммы положительные.
[PASS] Даты валидны (2026-08-04 09:08:31 — 2026-09-10 06:09:12).
[PASS] Полных дублей строк нет.


## 3. Базовые метрики на уровне строк

Это метрики "как есть", до склейки заказов — они будут пересчитаны ниже
после того, как определим, что считать одним заказом.

In [26]:
metrics = {
    "Строк в файле": len(df),
    "Уникальных student_id": df["student_id"].nunique(),
    "Курсов": df["course"].nunique(),
    "Период": f'{df["date"].min().date()} — {df["date"].max().date()}',
    "Min amount": df["amount"].min(),
    "Max amount": df["amount"].max(),
    "Средняя сумма строки": round(df["amount"].mean(),2),
    "Медианная сумма строки": df["amount"].median(),
    "Общая выручка": df["amount"].sum(),
}
pd.DataFrame(metrics.items(), columns=["Метрика", "Значение"])

,Метрика,Значение
0,Строк в файле,795
1,Уникальных student_id,606
2,Курсов,18
3,Период,2026-08-04 — 2026-09-10
4,Min amount,500.0
5,Max amount,19350.0
6,Средняя сумма строки,7427.26
7,Медианная сумма строки,7475.0
8,Общая выручка,5904671.67


## 4. Граница заказа: не пришлось подбирать вручную

Идея: посмотреть на разрыв во времени между соседними покупками одного и
того же человека.

In [15]:
df_sorted = df.sort_values(["student_id", "ts"])
gap_seconds = (
    df_sorted.groupby("student_id")["ts"]
    .apply(lambda s: s.sort_values().diff().dt.total_seconds())
    .dropna()
)

print(f"Всего пар соседних покупок одного человека: {len(gap_seconds)}")
print(f"Из них с разрывом ровно 0 секунд: {(gap_seconds == 0).sum()}")

non_zero = gap_seconds[gap_seconds > 0].sort_values()
print(f"Следующий по величине разрыв: {non_zero.iloc[0]:.0f} секунд "
      f"(~{non_zero.iloc[0] / 60:.0f} минут)")

Всего пар соседних покупок одного человека: 189
Из них с разрывом ровно 0 секунд: 167
Следующий по величине разрыв: 1589 секунд (~26 минут)


**Вывод.** Из 189 пар соседних покупок 167 совпадают до секунды, а
дальше сразу идёт разрыв со следующей покупкой по времени. Значит **заказ = все строки одного
`student_id` с одинаковым `ts`.**

In [17]:
orders = (
    df.groupby(["student_id", "ts"], as_index=False)
    .agg(amount=("amount", "sum"), n_courses=("course", "count"),
         courses=("course", lambda x: list(x)))
)
orders["date"] = orders["ts"].dt.normalize()

order_metrics = {
    "Заказов после склейки": len(orders),
    "Уникальных покупателей": orders["student_id"].nunique(),
    "Средний чек заказа": orders["amount"].mean(),
    "Медианный чек заказа": orders["amount"].median(),
}

def fmt(value):
    return f"{value:,.0f}".replace(",", " ")

pd.DataFrame(
    [(k, fmt(v)) for k, v in order_metrics.items()],
    columns=["Метрика", "Значение"],
)

,Метрика,Значение
0,Заказов после склейки,628
1,Уникальных покупателей,606
2,Средний чек заказа,9 402
3,Медианный чек заказа,8 950


## 5. Пакетные покупки — не исключение, а основная механика

Проверяем гипотезу из условия «иногда два курса продавались пакетом»:
для частых сумм заказа из двух курсов цена должна делиться на курсы
поровну - тогда это фиксированная цена пакета, а не случайное совпадение.

In [18]:
two_course_orders = orders[orders["n_courses"] == 2].copy()
package_sums = two_course_orders["amount"].value_counts().head(10)
print("Самые частые суммы заказов из двух курсов:")
print(package_sums)

print("\nДоля заказов больше чем с одним курсом:",
      f'{(orders["n_courses"] > 1).mean():.1%}')

Самые частые суммы заказов из двух курсов:
amount
14950.0    56
9900.0     24
13950.0    15
9990.0     14
16950.0     9
8990.0      2
19900.0     2
17950.0     2
12950.0     2
8900.0      2
Name: count, dtype: int64

Доля заказов больше чем с одним курсом: 24.4%


## 6. День недели — выходные решают всё

In [20]:
orders["weekday"] = orders["date"].dt.day_name()
orders["is_weekend"] = orders["date"].dt.dayofweek >= 5

weekday_stats = orders.groupby("is_weekend").agg(
    orders=("amount", "count"), revenue=("amount", "sum")
)
weekday_stats["orders_share"] = weekday_stats["orders"] / weekday_stats["orders"].sum()
weekday_stats["revenue_share"] = weekday_stats["revenue"] / weekday_stats["revenue"].sum()

weekday_display = weekday_stats.copy()
weekday_display.index = weekday_display.index.map({False: "Будни", True: "Выходные"})
weekday_display.index.name = "День"
weekday_display["revenue"] = weekday_display["revenue"].map(lambda v: f"{v:,.0f} ₽".replace(",", " "))
weekday_display["orders_share"] = weekday_display["orders_share"].map(lambda v: f"{v:.0%}")
weekday_display["revenue_share"] = weekday_display["revenue_share"].map(lambda v: f"{v:.0%}")
weekday_display.columns = ["Заказов", "Выручка", "Доля заказов", "Доля выручки"]
weekday_display

,Заказов,Выручка,Доля заказов,Доля выручки
День,,,,
Будни,327,3 215 866 ₽,52%,54%
Выходные,301,2 688 806 ₽,48%,46%


**Вывод.** Выходные — это два дня из семи, но на них приходится
почти половина заказов (48%) и почти половина выручки (45%). Доля по
выручке чуть ниже доли по заказам — то есть в выходные немного чаще
берут недорогие/одиночные покупки, а не дорогие пакеты, но разница
небольшая и общую картину не меняет.

## 7. Чего из этих данных узнать нельзя

1. **Откуда пришёл покупатель** — ни одного поля про источник трафика.
2. **Сколько потратили на рекламу** — исторический ROMI посчитать нельзя
   в принципе, знаменатель формулы заполнить нечем.
3. **Что было до 4 августа** — 38 дней истории, это мало для сезонности.
4. **Эффект скидки отдельно от эффекта выходного дня** — акции запускали
   именно по выходным, эти два эффекта переплетены и на этих данных не
   разделяются.
5. **Настоящее число повторных покупателей** — идентификатор в исходном
   процессе завязан на username, а его можно сменить.

## 8. Графики

In [21]:
daily_stats = (
    df.groupby("date")
    .agg(Total_Revenue=("amount", "sum"), Orders_Count=("student_id", "count"))
    .reset_index()
)

fig, ax1 = plt.subplots()
ax1.set_xlabel("Дата")
ax1.set_ylabel("Выручка (₽)", color="tab:blue")
ax1.plot(daily_stats["date"], daily_stats["Total_Revenue"], color="tab:blue",
         marker="o", linewidth=2, label="Выручка")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Количество заказов", color="tab:orange")
ax2.plot(daily_stats["date"], daily_stats["Orders_Count"], color="tab:orange",
         linestyle="--", marker="s", label="Заказы")
ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title("Динамика дневной выручки и объёма заказов")
fig.tight_layout()
plt.savefig("../reports/figures/01_daily.png")
plt.show()

C:\Users\s7omb\AppData\Local\Temp\ipykernel_15192\3275192844.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
course_stats = (
    df.groupby("course")
    .agg(Revenue=("amount", "sum"), Buyers=("student_id", "nunique"))
    .sort_values("Revenue", ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=course_stats, x="Revenue", y="course", hue="course", palette="viridis", legend=False, ax=ax)
plt.title("Выручка по курсам")
plt.xlabel("Суммарная выручка (₽)")
plt.ylabel("Курс")
for i, row in course_stats.iterrows():
    ax.text(row["Revenue"], i, f' {int(row["Revenue"]):,} ₽ ({row["Buyers"]} пок.)', va="center")
plt.tight_layout()
plt.savefig("../reports/figures/02_revenue_by_course.png")
plt.show()

C:\Users\s7omb\AppData\Local\Temp\ipykernel_15192\4069563629.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
plt.figure(figsize=(10, 5))
sns.histplot(df["amount"], bins=20, kde=True, color="teal")
plt.axvline(df["amount"].mean(), color="red", linestyle="--", label=f'Среднее: {df["amount"].mean():.0f} ₽')
plt.axvline(df["amount"].median(), color="orange", linestyle="-", label=f'Медиана: {df["amount"].median():.0f} ₽')
plt.title("Распределение сумм строк")
plt.xlabel("Сумма (₽)")
plt.ylabel("Количество строк")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/figures/03_amount_distribution.png")
plt.show()

C:\Users\s7omb\AppData\Local\Temp\ipykernel_15192\1252367734.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
df_sorted = df.sort_values(["student_id", "ts"])
df_sorted["next_course"] = df_sorted.groupby("student_id")["course"].shift(-1)
df_sorted["next_ts"] = df_sorted.groupby("student_id")["ts"].shift(-1)

naive_matrix = pd.crosstab(df_sorted["course"], df_sorted["next_course"], normalize="index")

# Настоящие повторные покупки: следующая строка — из ДРУГОГО заказа (другое ts)
real_transitions = df_sorted[df_sorted["ts"] != df_sorted["next_ts"]]
print(f"Всего пар курс->следующий курс: {df_sorted['next_course'].notna().sum()}")
print(f"Из них — настоящие повторные покупки (другой заказ): {real_transitions['next_course'].notna().sum()}")

plt.figure(figsize=(15, 15))
sns.heatmap(naive_matrix, annot=True, fmt=".1%", cmap="YlGnBu", cbar=True)
plt.title("Наивная матрица (включает состав пакетов — не читать как последовательность обучения)")
plt.xlabel("Следующий курс")
plt.ylabel("Первый курс")
plt.tight_layout()
plt.savefig("../reports/figures/04_naive_transition_matrix.png")
plt.show()

Всего пар курс->следующий курс: 189
Из них — настоящие повторные покупки (другой заказ): 22


C:\Users\s7omb\AppData\Local\Temp\ipykernel_15192\2581542089.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
